# BERTopic Parameter Tuning

Individual question-by-question BERTopic fitting with isolated parameter tuning.

Each question is fit separately with the same baseline parameters from the main `fit_bertopic()` function. This allows for targeted tuning of each question's topic model.

In [1]:
# Imports
import pandas as pd
import numpy as np

# BERTopic stack
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
import hdbscan
from umap import UMAP
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech


# Optional: Ollama/LLM support (uncomment if using)
import openai
from bertopic.representation import OpenAI

# Configure OpenAI for topic representation
client = openai.OpenAI(
    base_url="http://localhost:11434/v1",
    api_key='ollama'
)

llm_prompt = """
I have topic that contains the following documents: \n[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the above information, can you give a concise label of the topic using 3-6 words?
In your response, only provide the concise label without any additional explanation or formatting.
Additionally, please ensure that the label is descriptive of the underlying theme and uses most of the keywords provided.
"""

representation_model_llm = OpenAI(client, 
                              model='mistral')


/Users/tylerwiederich/Library/CloudStorage/OneDrive-UniversityofNebraska-Lincoln/4 - Obsidian Vault/Research/dissertation/ch2-experiential-learning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load data
df = pd.read_csv("../data/student-responses.csv")

# Filter only Bar Chart experiment responses
df = df[df["experiment"] == "Bar chart"]

print(f"Total responses: {len(df)}")
print(f"Dataframe shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Total responses: 654
Dataframe shape: (654, 21)
Columns: ['id', 'section_sis_id', 'section', 'attempt', 'What components of the experiment are clearer now than they were as a participant What questions do you still have for the experimenter Write 3 5 sentences reflecting on the abstract', 'Paste the response code you received after participating in the graphics experiment here', 'As of today I am at least 19 years of age', 'My instructor may share my reflection responses with the researchers in this study', 'What do you think the purpose of the experiment was', 'What elements of experimental design such as randomization or the use of a control group do you think were present in the experiment Why', 'What hypotheses might the experimenter have been testing', 'What sources of error are involved in this experiment', 'What variables were examined For each variable identify whether it was quantitative or categorical', 'In this class you ll be learning about the process of scientific investi

## Baseline BERTopic Parameters

All questions will be fit using these shared parameters. Modify here to tune globally, or override in individual question cells for targeted tuning.

In [3]:
# Initialize baseline models and configurations

# Embedding model
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# UMAP parameters
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# HDBSCAN parameters
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=3,
    min_samples=1,
    cluster_selection_method="eom",
    prediction_data=True
)

# Vectorizer model
vectorizer_model = CountVectorizer(
    stop_words="english",
    min_df=2,
    max_df=0.95
)

# C-TF-IDF model
ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

# Representation model (KeyBERT by default, change to OpenAI if using Ollama)
representation_model = KeyBERTInspired()

print("Baseline parameters initialized.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5700.69it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline parameters initialized.


# Global Settings

Each question will be structued as follows:

1. Fit a generic BERTopic
2. Update the model with different parameters UMAP and HDBSCAN

# Per-Question BERTopic Fitting

## Pre-Experiment

### Q1: In this class, you'll be learning about the process of scientific investigation...

**Column Index:** 13

In [ ]:
# Q1: Extract and clean responses
docs_q1 = df.iloc[:, 13].dropna().astype(str).str.strip()
docs_q1 = docs_q1[docs_q1.ne("")].tolist()

# Fit Initial BERTopic
model_q1 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=5, ngram_range=(1, 2), max_df=0.7),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.9),
        "POS": PartOfSpeech("en_core_web_sm"),
        "LLM": representation_model_llm
    },
    
    # Tunable models
    umap_model=UMAP(
        n_neighbors=10,
        n_components=10,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True #Required for probabilities
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q1, probs_q1 = model_q1.fit_transform(docs_q1)
model_q1.get_topic_info()

2026-04-14 13:27:11,480 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5613.85it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 20/20 [00:06<00:00,  2.94it/s]
2026-04-14 13:27:21,309 - BERTopic - Embedding - Completed ✓
2026-04-14 13:27:21,310 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 13:27:22,459 - BERTopic - Dimensionality - Completed ✓
2026-04-14 13:27:22,461 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 13:27:22,492 - BERTopic - Cluster - Completed ✓
2026-04-14 13:27:22,492 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,LLM,Representative_Docs
0,-1,172,-1_class_actually_ones_learning,"[class, actually, ones, learning, results thin...","[investigation perspective, public scientific,...","[class, actually, lots, data analyzing, comple...","[class, ones, learning, depth, lots, simple, a...",[Topic: Differences in scientific process],[From the perspective of a researcher I know t...
1,0,87,0_depth_researcher lot_think researcher_compar...,"[depth, researcher lot, think researcher, comp...","[public researchers, researchers perspective, ...","[depth, compared general, trying, parts, think...","[depth, parts, good, article, simple, final, f...",[Topic: Scientific investigation perspectives],"[Generally speaking, I believe that the perspe..."
2,1,49,1_individual_times_thought_importance,"[individual, times, thought, importance, easy,...","[investigation perspective, think researchers,...","[individual, importance, errors, hard, idea, d...","[individual, times, importance, easy, errors, ...",[Topic: Understanding scientific process perce...,"[Prior to this class, I was pretty familiar wi..."
3,2,45,2_contrast_peer review_news articles_ongoing,"[contrast, peer review, news articles, ongoing...","[understanding scientific, research process, p...","[contrast, peer review, journey, collect analy...","[contrast, ongoing, systematic, complexities, ...",[Scientific investigation process],[The process of scientific investigation invol...
4,3,42,3_news_public science_big_discoveries,"[news, public science, big, discoveries, trial...","[public science, view science, consuming scien...","[public science, big, figuring, step process, ...","[news, public science, big, discoveries, trial...",[Science perception gap],"[From the perspective of a researcher, the sci..."
5,4,40,4_peers_factor_conducted_field,"[peers, factor, conducted, field, actual, effo...","[investigation researchers, researchers perspe...","[peers, factor, conducted, actual, thought, ge...","[peers, factor, field, actual, good, effort, t...",[Topic: Perception of Scientific Research Proc...,[I think that research can take on many differ...
6,5,40,5_theory_reject_disprove_theories,"[theory, reject, disprove, theories, curiosity...","[believe scientific, using scientific, science...","[reject, individuals, causes, recording, creat...","[theory, theories, curiosity, solution, indivi...",[Scientific investigation process],[I think that science happens through the basi...
7,6,38,6_look like_stats_experts_investigation look,"[look like, stats, experts, investigation look...","[researches, think researchers, researchers po...","[look like, numbers, researches, drawn, true, ...","[stats, experts, numbers, researches, factual,...",[Scientific investigation steps],[From the perspective of a researcher learning...
8,7,38,7_group_researched_gathered_tested,"[group, researched, gathered, tested, answered...","[research process, hypothesis research, proces...","[answered, necessary, plan, doesn, population,...","[group, plan, necessary, population, control, ...",[Research methodology dynamics],[I think that the first part of the experiment...
9,8,29,8_interested_attention_regarding_results science,"[interested, attention, regarding, results sci...","[conduct research, researchers point, differen...","[interested, researchers point, life, pay, acc...","[interested, attention, eyes, ones, life, diff...",[Perspective of scientific research],[I feel the processes are both different. Bein...


In [5]:
model_q1.visualize_documents(docs_q1, hide_annotations=True)

In [6]:
model_q1.get_topic_info()

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,172,-1_class_actually_ones_learning,"[class, actually, ones, learning, results thin...","[investigation perspective, public scientific,...","[class, actually, lots, data analyzing, comple...","[class, ones, learning, depth, lots, simple, a...",[From the perspective of a researcher I know t...
1,0,87,0_depth_researcher lot_think researcher_compar...,"[depth, researcher lot, think researcher, comp...","[public researchers, researchers perspective, ...","[depth, compared general, trying, parts, think...","[depth, parts, good, article, simple, final, f...","[Generally speaking, I believe that the perspe..."
2,1,49,1_individual_times_thought_importance,"[individual, times, thought, importance, easy,...","[investigation perspective, think researchers,...","[individual, importance, errors, hard, idea, d...","[individual, times, importance, easy, errors, ...","[Prior to this class, I was pretty familiar wi..."
3,2,45,2_contrast_peer review_news articles_ongoing,"[contrast, peer review, news articles, ongoing...","[understanding scientific, research process, p...","[contrast, peer review, journey, collect analy...","[contrast, ongoing, systematic, complexities, ...",[The process of scientific investigation invol...
4,3,42,3_news_public science_big_discoveries,"[news, public science, big, discoveries, trial...","[public science, view science, consuming scien...","[public science, big, figuring, step process, ...","[news, public science, big, discoveries, trial...","[From the perspective of a researcher, the sci..."
5,4,40,4_peers_factor_conducted_field,"[peers, factor, conducted, field, actual, effo...","[investigation researchers, researchers perspe...","[peers, factor, conducted, actual, thought, ge...","[peers, factor, field, actual, good, effort, t...",[I think that research can take on many differ...
6,5,40,5_theory_reject_disprove_theories,"[theory, reject, disprove, theories, curiosity...","[believe scientific, using scientific, science...","[reject, individuals, causes, recording, creat...","[theory, theories, curiosity, solution, indivi...",[I think that science happens through the basi...
7,6,38,6_look like_stats_experts_investigation look,"[look like, stats, experts, investigation look...","[researches, think researchers, researchers po...","[look like, numbers, researches, drawn, true, ...","[stats, experts, numbers, researches, factual,...",[From the perspective of a researcher learning...
8,7,38,7_group_researched_gathered_tested,"[group, researched, gathered, tested, answered...","[research process, hypothesis research, proces...","[answered, necessary, plan, doesn, population,...","[group, plan, necessary, population, control, ...",[I think that the first part of the experiment...
9,8,29,8_interested_attention_regarding_results science,"[interested, attention, regarding, results sci...","[conduct research, researchers point, differen...","[interested, researchers point, life, pay, acc...","[interested, attention, eyes, ones, life, diff...",[I feel the processes are both different. Bein...


## Post-Experiment

### Q2: What do you think the purpose of the experiment was?

**Column Index:** 8

In [7]:
# Q2: Extract and clean responses
docs_q2 = df.iloc[:, 8].dropna().astype(str).str.strip()
docs_q2 = docs_q2[docs_q2.ne("")].tolist()

print(f"Q2 - Total responses: {len(docs_q2)}")

# Fit BERTopic
model_q2 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2), max_df=0.7),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm"),
        #"LLM": representation_model_llm
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=10,
        n_components=10,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=20,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True #Required for probabilities
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q2, probs_q2 = model_q2.fit_transform(docs_q2)
model_q2.get_topic_info()

Q2 - Total responses: 521


2026-04-14 08:47:35,647 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5698.74it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:02<00:00,  7.27it/s]
2026-04-14 08:47:40,547 - BERTopic - Embedding - Completed ✓
2026-04-14 08:47:40,547 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 08:47:41,293 - BERTopic - Dimensionality - Completed ✓
2026-04-14 08:47:41,294 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 08:47:41,309 - BERTopic - Cluster - Completed ✓
2026-04-14 08:47:41,310 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,47,-1_blocks_block_perspective_measure,"[blocks, block, perspective, measure, certain,...","[depth perception, people visualize, blocks, e...","[blocks, perspective, measure, perceive things...","[blocks, block, perspective, certain, tall, co...",[I was a little confused in the beginning of t...
1,0,144,0_hypothesis_taking_population_people read,"[hypothesis, taking, population, people read, ...","[experiment students, experiment understand, e...","[hypothesis, people read, questions, experimen...","[hypothesis, population, questions, question, ...",[Figuring out our problem solving skills. Mayb...
2,1,135,1_analyze_3d graphs_read_looking,"[analyze, 3d graphs, read, looking, people est...","[read graphs, graphs purpose, understand stati...","[analyze, 3d graphs, graphs easier, bar graphs...","[online, bars, reading, ways, screen, way, typ...",[The project aimed to investigate how students...
3,2,68,2_object_shapes_objects_things,"[object, shapes, objects, things, 3d models, l...","[experiment difference, experiment test, exper...","[shapes, objects, 3d models, larger, depth per...","[object, shapes, objects, things, percentage, ...",[I think the purpose of the experiment was to ...
4,3,53,3_bar graphs_bar graph_bars_values,"[bar graphs, bar graph, bars, values, heights,...","[bar graphs, bar graph, difference bars, size ...","[bar graphs, bar graph, graphs think, 3d bar, ...","[bars, values, heights, bar, perspectives, com...",[I think the purpose of the experiment was to ...
5,4,28,4_images_graphs 3d_experiment different_2d graphs,"[images, graphs 3d, experiment different, 2d g...","[experiment understand, experiment difference,...","[graphs 3d, 2d graphs, visualization, affect p...","[images, visualization, different ways, beginn...","[At the beginning of the experiment, I was com..."
6,5,26,5_3d printed_printed_charts_digital 3d,"[3d printed, printed, charts, digital 3d, 2d d...","[3d charts, 3d graphs, charts purpose, 3d bar,...","[3d printed, charts, 3d digital, 3d charts, ba...","[charts, perceptual, visuals, researchers, typ...",[This study aimed to:\n\n*\n\nReplicate and ex...
7,6,20,6_experiment likely_people perception_context_...,"[experiment likely, people perception, context...","[depth perception, perception size, people per...","[experiment likely, people perception, 3d mode...","[context, objects, proportion, relative, answe...",[The purpose of this experiment is to see how ...


In [8]:
model_q2.visualize_documents(docs_q2, hide_annotations=True)

### Q3: What hypotheses might the experimenter have been testing?

**Column Index:** 10

In [9]:
# Q3: Extract and clean responses
docs_q3 = df.iloc[:, 10].dropna().astype(str).str.strip()
docs_q3 = docs_q3[docs_q3.ne("")].tolist()

print(f"Q3 - Total responses: {len(docs_q3)}")

# Fit BERTopic
model_q3 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2), max_df=0.7),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm"),
        #"LLM": representation_model_llm
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=18,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True #Required for probabilities
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q3, probs_q3 = model_q3.fit_transform(docs_q3)
model_q3.get_topic_info()

Q3 - Total responses: 517


2026-04-14 08:47:46,752 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11225.12it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:02<00:00,  6.76it/s]
2026-04-14 08:47:51,022 - BERTopic - Embedding - Completed ✓
2026-04-14 08:47:51,022 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 08:47:51,864 - BERTopic - Dimensionality - Completed ✓
2026-04-14 08:47:51,864 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 08:47:51,881 - BERTopic - Cluster - Completed ✓
2026-04-14 08:47:51,882 - BERTopic - Representation - Extracting topics using c-TF-IDF for to

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,135,-1_color_bar graphs_colors_bigger,"[color, bar graphs, colors, bigger, reading, 5...","[bar graphs, bar graph, interpret data, graphs...","[color, bar graphs, bigger, reading, values, s...","[color, colors, bigger, values, stats, questio...",[They might have hyponthesized that participan...
1,0,151,0_experiment_based_shapes_estimations,"[experiment, based, shapes, estimations, relat...","[hypothesis students, hypotheses experimenter,...","[shapes, experiment testing, sizes, object, gr...","[experiment, shapes, estimations, relative, th...",[The experiment might have been testing whethe...
2,1,149,1_charts_digital_printed_3d printed,"[charts, digital, printed, 3d printed, physica...","[compared 3d, 3d graphs, bar graphs, graphs 3d...","[charts, 3d printed, 3d graphs, 2d graphs, per...","[charts, digital, physical, chart, screen, per...",[The experimenters likely had several hypothes...
3,2,29,2_bars_bar graph_tall_estimation,"[bars, bar graph, tall, estimation, closer, he...","[bars compared, bar height, size bar, bar grap...","[bar graph, estimation, heights bar, size diff...","[bars, tall, closer, estimation, angles, apart...",[If smaller bars in a bar graph are next to la...
4,3,22,3_statistics_taking_class_stats,"[statistics, taking, class, stats, numerical, ...","[statistic, hypothesis students, statistics, s...","[statistics, statistic, unl students, students...","[statistics, class, stats, numerical, statisti...",[STAT 218 students will guess the correct pres...
5,4,16,4_null hypothesis_alternative_alternative hypo...,"[null hypothesis, alternative, alternative hyp...","[null hypothesis, testing hypothesis, alternat...","[null hypothesis, graphs 3d, difference accura...","[null hypothesis, alternative, alternative hyp...",[The null hypothesis the experimenter may have...
6,5,15,5_blocks_taller_believe_think experimenter,"[blocks, taller, believe, think experimenter, ...","[testing students, students accurately, blocks...","[think experimenter, estimate proportions, ski...","[blocks, taller, skills, degree, shapes, lengt...",[Students are better able to estimate proporti...


In [10]:
model_q3.visualize_documents(docs_q3, hide_annotations=True)

### Q4: What sources of error are involved in this experiment?

**Column Index:** 11

In [11]:
# Q4: Extract and clean responses
docs_q4 = df.iloc[:, 11].dropna().astype(str).str.strip()
docs_q4 = docs_q4[docs_q4.ne("")].tolist()

print(f"Q4 - Total responses: {len(docs_q4)}")

# Fit BERTopic
model_q4 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2), max_df=0.7),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm"),
        #"LLM": representation_model_llm
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=18,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q4, probs_q4 = model_q4.fit_transform(docs_q4)
model_q4.get_topic_info()

Q4 - Total responses: 517


2026-04-14 08:47:59,784 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 44468.11it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:02<00:00,  5.81it/s]
2026-04-14 08:48:04,922 - BERTopic - Embedding - Completed ✓
2026-04-14 08:48:04,923 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 08:48:05,819 - BERTopic - Dimensionality - Completed ✓
2026-04-14 08:48:05,819 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 08:48:05,839 - BERTopic - Cluster - Completed ✓
2026-04-14 08:48:05,840 - BERTopic - Representation - Extracting topics using c-TF-IDF for to

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,129,-1_numbers_shapes_online_slider,"[numbers, shapes, online, slider, compared, pi...","[error participants, did experiment, error inv...","[numbers, shapes, slider, compared, pieces, er...","[numbers, shapes, online, slider, pieces, shap...",[Sources of error involved in this experiment ...
1,0,163,0_lack_error involved_lot_include,"[lack, error involved, lot, include, wrong, an...","[sample size, experiment include, experiment p...","[lack, error involved, environmental, survey, ...","[lack, lot, wrong, answers, times, environment...",[A source of error that could have been involv...
2,1,62,1_3d_charts_3d printed_models,"[3d, charts, 3d printed, models, printed, angl...","[experiment people, charts, printed graphs, me...","[3d, charts, 3d printed, models, bar heights, ...","[3d, charts, models, angle, depth, objects, bi...",[There are a few ways errors could creep into ...
3,2,51,2_graph_graphs_answers_understand,"[graph, graphs, answers, understand, experienc...","[graph, graphs, bar graph, error people, stati...","[graphs, answers, experience, participating, r...","[graph, graphs, answers, experience, values, g...",[The error is just human guessing. One thing I...
4,3,38,3_bar_bigger_know_smaller,"[bar, bigger, know, smaller, heights, bars, in...","[measurement errors, measurement, perception d...","[heights, incorrectly, measurement, visual per...","[bar, bigger, smaller, heights, bars, measurem...",[1. Subjective perception - we may perceive th...
5,4,23,4_statistics_sampling_218_random sampling,"[statistics, sampling, 218, random sampling, c...","[sample size, sample bias, sampling error, sta...","[statistics, population, statistics students, ...","[statistics, sampling, random sampling, class,...",[The sampling group may be biased and not repr...
6,5,19,5_blocks_id_internet_sets,"[blocks, id, internet, sets, previous, correct...","[blocks, different results, results different,...","[blocks, id, internet, previous, different res...","[blocks, previous, sets, internet, correct, li...",[I guess it could be something like we all did...
7,6,17,6_human perception_eyesight_comparisons_problems,"[human perception, eyesight, comparisons, prob...","[visual impairments, eye sight, eyesight, impa...","[human perception, eyesight, comparisons, subj...","[eyesight, visual impairments, subjective, com...",[Eye sight impairment such as wearing glasses ...
8,7,15,7_color_likely_screen_potentially,"[color, likely, screen, potentially, brightnes...","[perception differences, sample size, visual i...","[brightness, depth perception, visual impairme...","[color, likely, screen, brightness, quality, p...",[Some sources of error could possibly include ...


### Q5: What variables were examined? For each variable, identify whether it was quantitative or categorical.

**Column Index:** 12

In [12]:
# Q5: Extract and clean responses
docs_q5 = df.iloc[:, 12].dropna().astype(str).str.strip()
docs_q5 = docs_q5[docs_q5.ne("")].tolist()

print(f"Q5 - Total responses: {len(docs_q5)}")

# Fit BERTopic
model_q5 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2), max_df=0.7),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm"),
        #"LLM": representation_model_llm
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=18,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q5, probs_q5 = model_q5.fit_transform(docs_q5)
model_q5.get_topic_info()

Q5 - Total responses: 515


2026-04-14 08:48:09,550 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11639.96it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:02<00:00,  6.82it/s]
2026-04-14 08:48:13,735 - BERTopic - Embedding - Completed ✓
2026-04-14 08:48:13,735 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 08:48:14,607 - BERTopic - Dimensionality - Completed ✓
2026-04-14 08:48:14,608 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 08:48:14,624 - BERTopic - Cluster - Completed ✓
2026-04-14 08:48:14,625 - BERTopic - Representation - Extracting topics using c-TF-IDF for to

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,90,-1_dependent_groups_vs 3d_models,"[dependent, groups, vs 3d, models, height quan...","[graphs categorical, graph quantitative, quant...","[dependent, groups, vs 3d, height quantitative...","[dependent, groups, models, people, independen...",[The height of the bar graph - it would be qua...
1,0,62,0_variable type_variable examined_type graph_v...,"[variable type, variable examined, type graph,...","[graphs quantitative, graph quantitative, quan...","[variable examined, type graph, variable categ...","[categorical variable, graphs, computer, quant...",[Some variables in this experiment may include...
2,1,56,1_shape_person_categorical size_picked,"[shape, person, categorical size, picked, tall...","[height categorical, quantitative variable, he...","[categorical size, taller shorter, independent...","[shape, person, explanatory, taller, independe...",[The time it takes to identify the taller one ...
3,2,52,2_quantitative bar_categorical smaller_smaller...,"[quantitative bar, categorical smaller, smalle...","[bar compared, bars quantitative, categorical ...","[bars quantitative, quantitative height, bar c...","[smaller bar, larger bar, percent, shorter, hi...",[Which bar is smaller - categorical\n\nhow muc...
4,3,43,3_qualitative_like_models_questions,"[qualitative, like, models, questions, charts,...","[variables quantitative, qualitative variables...","[qualitative, charts, variables categorical, q...","[qualitative, models, questions, charts, able,...",[One variable is what kit the participant was ...
5,4,40,4_quantitative graph_graph categorical_reading...,"[quantitative graph, graph categorical, readin...","[graphs quantitative, graph quantitative, grap...","[graphs quantitative, larger categorical, quan...","[reading, graphs, different types, percentages...",[What percentage of a bar compared to another ...
6,5,40,5_columns_say_examined bar_numbers,"[columns, say, examined bar, numbers, determin...","[variables quantitative, categorical quantitat...","[columns, examined bar, experiment, variable q...","[columns, numbers, experiment, letter, second,...",[The variables examined were which bar was hig...
7,6,37,6_circle triangle_triangle circle_units_taller,"[circle triangle, triangle circle, units, tall...","[quantitative categorical, categorical variabl...","[triangle categorical, quantitative categorica...","[units, taller, ratios, line, tall, second, sh...",[We first decided what tower was taller (wheth...
8,7,37,7_online_correctly categorical_number quantita...,"[online, correctly categorical, number quantit...","[students quantitative, questions quantitative...","[correctly categorical, number quantitative, c...","[online, correct answers, percent, proportion,...",[how accurate we guessed on the online or in p...
9,8,31,8_ratio_error_chart_3d digital,"[ratio, error, chart, 3d digital, digital 3d, ...","[chart type, bar chart, bar graphs, chart cate...","[3d digital, chart type, ratio quantitative, m...","[ratio, error, chart, separated, adjacent, dig...",[The experiment looked at a mix of chart featu...


### Q6: What elements of experimental design, such as randomization or the use of a control group, do you think were present in the experiment? Why?

**Column Index:** 9

In [13]:
# Q6: Extract and clean responses
docs_q6 = df.iloc[:, 9].dropna().astype(str).str.strip()
docs_q6 = docs_q6[docs_q6.ne("")].tolist()

print(f"Q6 - Total responses: {len(docs_q6)}")

# Fit BERTopic
model_q6 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=1, ngram_range=(1, 2), max_df=1.0),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm"),
        #"LLM": representation_model_llm
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=18,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=12,
        min_samples=2,
        cluster_selection_method="eom",
        prediction_data=True
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q6, probs_q6 = model_q6.fit_transform(docs_q6)
model_q6.get_topic_info()

Q6 - Total responses: 514


2026-04-14 08:48:18,458 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15434.78it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:03<00:00,  4.93it/s]
2026-04-14 08:48:25,144 - BERTopic - Embedding - Completed ✓
2026-04-14 08:48:25,145 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 08:48:26,002 - BERTopic - Dimensionality - Completed ✓
2026-04-14 08:48:26,002 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 08:48:26,023 - BERTopic - Cluster - Completed ✓
2026-04-14 08:48:26,023 - BERTopic - Representation - Extracting topics using c-TF-IDF for to

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,61,-1_design_measures_repeated measures_computer,"[design, measures, repeated measures, computer...","[experimental design, experiment randomization...","[repeated measures, randomized, bias, people, ...","[design, measures, computer, present, randomiz...",[I think the extra strips were in there to see...
1,0,144,0_control group_control_group_students,"[control group, control, group, students, 218,...","[experimental group, control group, controlled...","[control group, students, stats, think, use co...","[control, group, students, stats, class, exper...",[Since the experiment was limited to only stat...
2,1,97,1_graphs_graph_bar_bars,"[graphs, graph, bar, bars, bar graphs, order, ...","[randomization used, used randomization, graph...","[graph, bar graphs, graphs different, think ra...","[graphs, graph, bar, bars, order, 2d, differen...",[Researchers may have randomly assigned partic...
3,2,60,2_students_study_experimental design_element,"[students, study, experimental design, element...","[experiment randomization, randomization used,...","[experimental design, participants, experiment...","[students, study, experimental design, element...",[The randomization aspect could be seen in the...
4,3,30,3_questions_kit_kits_pool,"[questions, kit, kits, pool, randomization use...","[randomization used, randomized designed, rand...","[randomization used, questions randomized, dif...","[questions, kit, kits, pool, random questions,...",[i think this design was mainly randomization ...
5,4,26,4_blocks_set blocks_bag_block,"[blocks, set blocks, bag, block, chose, bags, ...","[random blocks, randomization, groups random, ...","[set blocks, group random, groups, audience, c...","[blocks, bag, bags, good, certain, groups, set...",[I guess overall they were pretty random becau...
6,5,24,5_charts_chart_2d_bar,"[charts, chart, 2d, bar, differences, 3d, prin...","[2d charts, bar charts, chart types, charts ra...","[chart, 3d, chart types, 2d charts, 3d printed...","[charts, chart, bar, differences, 3d, types, r...",[The experiment included several good design e...
7,6,21,6_bags_bag_pick_bag different,"[bags, bag, pick, bag different, picked, bag t...","[randomization bags, random bags, randomizatio...","[randomization bags, random fact, random bags,...","[bags, bag, numbered, random bags, different b...",[I think randomization of the bags that we got...
8,7,21,7_stats_courses_random_random sampling,"[stats, courses, random, random sampling, samp...","[random sample, randomly sampled, random assig...","[random sampling, played experiment, courses c...","[stats, courses, random, random sampling, samp...",[In the experiment there was random sampling b...
9,8,17,8_cows_intervention_confounding_randomization ...,"[cows, intervention, confounding, randomizatio...","[experimental treatment, randomization helps, ...","[intervention, confounding variables, helps en...","[cows, intervention, variables, old, years, va...","[Generally, elements like randomization help e..."


## Abstract Reflection

### Q7: What components of the experiment are clearer now than they were as a participant? What questions do you still have for the experimenter?

**Column Index:** 4

In [14]:
# Q7: Extract and clean responses
docs_q7 = df.iloc[:, 4].dropna().astype(str).str.strip()
docs_q7 = docs_q7[docs_q7.ne("")].tolist()

print(f"Q7 - Total responses: {len(docs_q7)}")

# Fit BERTopic
model_q7 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=1, ngram_range=(1, 2), max_df=1.0),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm"),
        #"LLM": representation_model_llm
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=22,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=18,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q7, probs_q7 = model_q7.fit_transform(docs_q7)
model_q7.get_topic_info()

Q7 - Total responses: 495


2026-04-14 08:48:31,088 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5597.28it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 16/16 [00:06<00:00,  2.46it/s]
2026-04-14 08:48:39,671 - BERTopic - Embedding - Completed ✓
2026-04-14 08:48:39,671 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 08:48:40,575 - BERTopic - Dimensionality - Completed ✓
2026-04-14 08:48:40,576 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 08:48:40,595 - BERTopic - Cluster - Completed ✓
2026-04-14 08:48:40,596 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,107,-1_bar_charts_people_graphs,"[bar, charts, people, graphs, bar charts, prin...","[3d graphs, bar charts, bar graph, graphs, cha...","[charts, people, graphs, bar charts, printed, ...","[bar, charts, people, graphs, different, resul...","[After reading the abstract and methods, sever..."
1,0,71,0_experiment_doing_questions_clearer,"[experiment, doing, questions, clearer, variab...","[understand experiment, experiment clearer, ov...","[experiment, variables, abstract, design, purp...","[experiment, questions, clearer, variables, st...",[The background of the experiment is much clea...
2,1,51,1_graph_3d graphs_2d_graphs,"[graph, 3d graphs, 2d, graphs, chart, 3d, diff...","[3d charts, 3d graphs, 3d chart, 3d vs, 3d gra...","[3d graphs, graphs, 3d charts, 2d graphs, diff...","[graph, 2d, graphs, chart, 3d, difference, cha...","[The exact purpose of this experiment, to dete..."
3,2,46,2_models_depth_objects_2d,"[models, depth, objects, 2d, depth perception,...","[depth perception, visual depth, affect depth,...","[depth, depth perception, 3d models, experimen...","[models, depth, objects, 2d, depth perception,...","[After reading the abstract, it is now much cl..."
4,3,42,3_charts_bar_heights_3d printed,"[charts, bar, heights, 3d printed, printed, ba...","[3d charts, bar charts, clearer experiment, ba...","[charts, 3d printed, bar heights, printed char...","[charts, bar, heights, digital, 3d, 2d, ratios...","[Reflecting on the abstract, it’s now much cle..."
5,4,42,4_graphs_2d graphs_3d graphs_different,"[graphs, 2d graphs, 3d graphs, different, kit,...","[experiment graphs, experiment clearer, readin...","[graphs, 2d graphs, 3d graphs, kit, types grap...","[graphs, different, kit, people, able, compute...","[When participating in the experiment, I hones..."
6,5,40,5_visualization_perceptual_judgments_3d printed,"[visualization, perceptual, judgments, 3d prin...","[bar charts, charts compared, chart types, vis...","[visualization, 3d printed, effectiveness, bar...","[visualization, perceptual, judgments, charts,...","[As a subject in the study, I did not know why..."
7,6,36,6_graphs_know_responses_just,"[graphs, know, responses, just, clearer partic...","[experiment clearer, reading graphs, measured ...","[clearer participant, abstract, clearer, respo...","[graphs, responses, experiment, components, ab...",[What is clearer now than it was when I was a ...
8,7,20,7_models_online_printed_3d models,"[models, online, printed, 3d models, shape, pr...","[printed 3d, study 3d, 3d models, 3d, printed ...","[3d models, shape, printed 3d, 3d printed, com...","[models, online, shape, model, image, ones, 3d...",[I now understand the difference between the 2...
9,8,20,8_bars_heights_claim_supposed,"[bars, heights, claim, supposed, different hei...","[experiment clearer, purpose experiment, matte...","[different heights, experiment, questions expe...","[bars, heights, claim, different heights, clea...",[The component of the color corresponding to t...


## Presentation Reflection

### Q8: How did the information you gained from the components of this project (participation, post-study reflection, extended abstract, presentation) differ?

**Column Index:** 14

In [15]:
# Q8: Extract and clean responses
docs_q8 = df.iloc[:, 14].dropna().astype(str).str.strip()
docs_q8 = docs_q8[docs_q8.ne("")].tolist()

print(f"Q8 - Total responses: {len(docs_q8)}")

# Fit BERTopic
model_q8 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=1, ngram_range=(1, 2), max_df=1.0),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm"),
        #"LLM": representation_model_llm
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=12,
        min_samples=2,
        cluster_selection_method="eom",
        prediction_data=True
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q8, probs_q8 = model_q8.fit_transform(docs_q8)
model_q8.get_topic_info()

Q8 - Total responses: 405


2026-04-14 08:48:47,862 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4954.30it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:04<00:00,  2.89it/s]
2026-04-14 08:48:54,790 - BERTopic - Embedding - Completed ✓
2026-04-14 08:48:54,791 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 08:48:55,432 - BERTopic - Dimensionality - Completed ✓
2026-04-14 08:48:55,433 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 08:48:55,446 - BERTopic - Cluster - Completed ✓
2026-04-14 08:48:55,447 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,61,-1_learned_did_lot_better,"[learned, did, lot, better, information learne...","[experiment presentation, study reflection, po...","[learned, information learned, learn, experime...","[lot, better, able, information, experiment, c...",[I feel at every step I gained a little more i...
1,0,99,0_3d_2d_graphs_graph,"[3d, 2d, graphs, graph, 3d graphs, charts, pri...","[3d charts, 3d graphs, bar charts, 3d graph, 2...","[3d, 3d graphs, perception, 2d graphs, 3d prin...","[3d, 2d, graphs, graph, charts, perception, ac...",[Participation in the project involves activel...
2,1,34,1_experimental_statistical_design_component,"[experimental, statistical, design, component,...","[study design, understanding experiment, purpo...","[reflecting, experiment, experiment focused, p...","[experimental, statistical, design, component,...",[Every element presented a distinct level of c...
3,2,33,2_study reflection_post study_post_reflection,"[study reflection, post study, post, reflectio...","[study design, participant study, study implic...","[study reflection, post study, extended abstra...","[study reflection, post, reflection, participa...",[The participation piece came first so I did n...
4,3,33,3_experiment_differed_video_information reflected,"[experiment, differed, video, information refl...","[understand experiment, experiment information...","[experiment, video, information reflected, exp...","[experiment, video, round, finish, credit, for...",[The information I gained from this is a lot m...
5,4,32,4_components_gained components_components proj...,"[components, gained components, components pro...","[project differed, project differ, differed pa...","[gained components, differed, components diffe...","[components, project, information, class, part...",[The information I gained from the components ...
6,5,31,5_participation_allowed_nice_times,"[participation, allowed, nice, times, differen...","[abstract participation, study presentation, a...","[participation, gave, abstract presentation, g...","[participation, nice, times, different, abstra...","[They all differ, because each time we got a l..."
7,6,21,6_informative_presentation informative_explain...,"[informative, presentation informative, explai...","[study abstract, participated abstract, study ...","[informative, presentation informative, simple...","[informative, abstract, simpler, brief, presen...",[During the participation and post-study refle...
8,7,18,7_helped_project_writing_ve,"[helped, project, writing, ve, reflection thin...","[presenting project, project taught, participa...","[reflection think, presentation helped, ideas,...","[project, view, things, ideas, experience, app...",[*\n\nParticipation: This aspect involved acti...
9,8,15,8_kits_different kits_like_confusing,"[kits, different kits, like, confusing, conduc...","[presentation gave, insight presentation, cond...","[different kits, cause like, knowledge gaining...","[kits, different kits, confusing, different in...",[The experiment was more hands on and I didn't...


### Q9: What components were emphasized in the presentation that weren't emphasized in the abstract? Why do you think that is?

**Column Index:** 16

In [16]:
# Q9: Extract and clean responses
docs_q9 = df.iloc[:, 16].dropna().astype(str).str.strip()
docs_q9 = docs_q9[docs_q9.ne("")].tolist()

print(f"Q9 - Total responses: {len(docs_q9)}")

# Fit BERTopic
model_q9 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=1, ngram_range=(1, 2), max_df=1.0),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm"),
        #"LLM": representation_model_llm
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=12,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=10,
        min_samples=2,
        cluster_selection_method="eom",
        prediction_data=True
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q9, probs_q9 = model_q9.fit_transform(docs_q9)
model_q9.get_topic_info()

Q9 - Total responses: 404


2026-04-14 08:49:01,595 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6681.88it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:02<00:00,  4.36it/s]
2026-04-14 08:49:07,117 - BERTopic - Embedding - Completed ✓
2026-04-14 08:49:07,117 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 08:49:07,707 - BERTopic - Dimensionality - Completed ✓
2026-04-14 08:49:07,708 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 08:49:07,724 - BERTopic - Cluster - Completed ✓
2026-04-14 08:49:07,724 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,1,-1_written text_things easier_easier demonstra...,"[written text, things easier, easier demonstra...","[presentation abstract, presentation emphasize...","[written text, easier demonstrate, cons using,...","[pros, previous studies, prevalent, text, expl...","[In the presentation, they emphasized the resu..."
1,0,222,0_abstract_study_results_presentation,"[abstract, study, results, presentation, exper...","[abstract presentation, presentation abstract,...","[presentation, experiment, emphasized presenta...","[abstract, study, results, presentation, exper...",[The presentation placed more emphasis on the ...
2,1,154,1_3d_graphs_2d_3d graphs,"[3d, graphs, 2d, 3d graphs, emphasized, presen...","[presentation abstract, presentation emphasize...","[graphs, 3d graphs, abstract, graphics, data, ...","[3d, graphs, 2d, presentation, graph, abstract...",[Some components that were emphasized in the p...
3,2,27,2_audience_visual_presentations_aids,"[audience, visual, presentations, aids, visual...","[presentation emphasized, abstract presentatio...","[presentations, visual aids, engaging, engage ...","[audience, visual, presentations, aids, visual...",[It emphasized the visual and interactive aspe...


### Q10: What critiques do you have of this study and its design? What would have made the study better?

**Column Index:** 17

In [17]:
# Q10: Extract and clean responses
docs_q10 = df.iloc[:, 17].dropna().astype(str).str.strip()
docs_q10 = docs_q10[docs_q10.ne("")].tolist()

print(f"Q10 - Total responses: {len(docs_q10)}")

# Fit BERTopic
model_q10 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2), max_df=0.7),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm"),
        #"LLM": representation_model_llm
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=18,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True #Required for probabilities
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q10, probs_q10 = model_q10.fit_transform(docs_q10)
model_q10.get_topic_info()

Q10 - Total responses: 405


2026-04-14 08:49:10,103 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9445.24it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:02<00:00,  4.66it/s]
2026-04-14 08:49:15,561 - BERTopic - Embedding - Completed ✓
2026-04-14 08:49:15,562 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 08:49:16,208 - BERTopic - Dimensionality - Completed ✓
2026-04-14 08:49:16,209 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 08:49:16,221 - BERTopic - Cluster - Completed ✓
2026-04-14 08:49:16,222 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,48,-1_long_end_large_thing say,"[long, end, large, thing say, assigned, partic...","[purpose experiment, participate study, study ...","[long, purpose experiment, having participants...","[long, end, large, multiple, sampling, compone...","[If possible, to obtain a more diverse group, ..."
1,0,150,0_bar_ve_color_types,"[bar, ve, color, types, person, easier, didn, ...","[results critique, study improved, statistics,...","[color, types, easier, affect, physical, diffe...","[bar, color, types, person, easier, confusing,...",[One critique is that the study mostly used pa...
2,1,90,1_confused_abstract_given_explain,"[confused, abstract, given, explain, overall, ...","[study designed, results study, purpose study,...","[abstract, helpful, experiment better, instruc...","[confused, abstract, overall, helpful, instruc...",[I believe the study is well-designed and cann...
3,2,50,2_statistics_department_general_outside,"[statistics, department, general, outside, lar...","[study designed, study used, study students, r...","[statistics, department, larger sample, stats ...","[statistics, department, general, larger sampl...",[I feel like one could lack some randomization...
4,3,50,3_project_don critiques_make better_critiques ...,"[project, don critiques, make better, critique...","[participate study, critiques think, study did...","[project, don critiques, make better, critique...","[project, specific, easy, life, minutes, abstr...",[I would say that one critique I have of the s...
5,4,17,4_control_findings_additionally_validity,"[control, findings, additionally, validity, di...","[study used, results study, results critique, ...","[validity, diversity, generalize, diverse, bia...","[control, findings, validity, potential, diver...","[While the study provided valuable insights, i..."


### Q11: If you had to hear about this study using only the extended abstract or only the presentation, which one would you prefer? Which one would be better for determining whether the experiment was well designed?

**Column Index:** 15

In [18]:
# Q11: Extract and clean responses
docs_q11 = df.iloc[:, 15].dropna().astype(str).str.strip()
docs_q11 = docs_q11[docs_q11.ne("")].tolist()

print(f"Q11 - Total responses: {len(docs_q11)}")

# Fit BERTopic
model_q11 = BERTopic(
    # Fixed models
    embedding_model='all-mpnet-base-v2',
    ctfidf_model=ctfidf_model,
    vectorizer_model = CountVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2), max_df=0.7),
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=0.3),
        "POS": PartOfSpeech("en_core_web_sm"),
        #"LLM": representation_model_llm
    },

    # Tunable models
    umap_model=UMAP(
        n_neighbors=18,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    ),
    hdbscan_model=hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=3,
        cluster_selection_method="eom",
        prediction_data=True #Required for probabilities
    ),

    # Other parameters
    verbose=True,
    nr_topics="auto",
    calculate_probabilities=True
)

topics_q11, probs_q11 = model_q11.fit_transform(docs_q11)
model_q11.get_topic_info()

Q11 - Total responses: 405


2026-04-14 08:49:20,604 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6133.06it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 13/13 [00:02<00:00,  4.83it/s]
2026-04-14 08:49:25,744 - BERTopic - Embedding - Completed ✓
2026-04-14 08:49:25,745 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 08:49:26,393 - BERTopic - Dimensionality - Completed ✓
2026-04-14 08:49:26,394 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 08:49:26,407 - BERTopic - Cluster - Completed ✓
2026-04-14 08:49:26,408 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,4,-1_presentation goes_experimental design_exper...,"[presentation goes, experimental design, exper...","[design presentation, use presentation, presen...","[presentation goes, experimental design, exper...","[experimental design, experimental, support, v...",[I would prefer the presentation. It goes into...
1,0,242,0_abstract better_extended abstract_extended_p...,"[abstract better, extended abstract, extended,...","[abstract presentation, abstract better, exten...","[abstract better, abstract presentation, deter...","[extended abstract, extended, abstract present...",[If I had to choose between hearing about the ...
2,1,58,1_person_people_designed prefer_lot sense,"[person, people, designed prefer, lot sense, l...","[preferred presentation, better presentation, ...","[people, designed prefer, determining experime...","[person, people, lot, actual, paper, things, r...",[I would definitely prefer the presentation. H...
3,2,58,2_step_learn better_able_watch,"[step, learn better, able, watch, learner, vis...","[preferred presentation, watch presentation, r...","[step, learn better, learner, visual learner, ...","[step, able, learner, visual learner, talk, cl...","[The presentation. Like I said before, I don't..."
4,3,23,3_bar_actually_believe_paper,"[bar, actually, believe, paper, presentation v...","[preferred presentation, presentation prefer, ...","[bar, paper, understand abstract, graph, prese...","[bar, paper, hard, reader, graph, difficult, l...",[I prefer the presentation. I think it does a ...
5,4,20,4_presentation gave_makes easier_watching_hear...,"[presentation gave, makes easier, watching, he...","[presentation prefer, presentation thought, pr...","[presentation gave, hear study, presentation e...","[results, questions, charts, different, visual...",[I would say that the presentation is the one ...


## Store Per-Question Artifacts

Create a dictionary to organize all models, topics, probabilities, and document sets for downstream analysis and tuning.

In [19]:
# Organize results for easy access and tuning
results = {
    'q1': {'model': model_q1, 'topics': topics_q1, 'probs': probs_q1, 'docs': docs_q1},
    'q2': {'model': model_q2, 'topics': topics_q2, 'probs': probs_q2, 'docs': docs_q2},
    'q3': {'model': model_q3, 'topics': topics_q3, 'probs': probs_q3, 'docs': docs_q3},
    'q4': {'model': model_q4, 'topics': topics_q4, 'probs': probs_q4, 'docs': docs_q4},
    'q5': {'model': model_q5, 'topics': topics_q5, 'probs': probs_q5, 'docs': docs_q5},
    'q6': {'model': model_q6, 'topics': topics_q6, 'probs': probs_q6, 'docs': docs_q6},
    'q7': {'model': model_q7, 'topics': topics_q7, 'probs': probs_q7, 'docs': docs_q7},
    'q8': {'model': model_q8, 'topics': topics_q8, 'probs': probs_q8, 'docs': docs_q8},
    'q9': {'model': model_q9, 'topics': topics_q9, 'probs': probs_q9, 'docs': docs_q9},
    'q10': {'model': model_q10, 'topics': topics_q10, 'probs': probs_q10, 'docs': docs_q10},
    'q11': {'model': model_q11, 'topics': topics_q11, 'probs': probs_q11, 'docs': docs_q11},
}

print("All models fitted and stored in 'results' dictionary.")
print("\nAccess individual results like: results['q1']['model'].get_topic_info()")

All models fitted and stored in 'results' dictionary.

Access individual results like: results['q1']['model'].get_topic_info()


## Next Steps: Tuning Individual Questions

To tune parameters for a specific question:

1. Modify the baseline parameters in the **Baseline BERTopic Parameters** section
2. Or override parameters in an individual question cell (create a local `umap_model`, `hdbscan_model`, etc.)
3. Re-run the question cell to refit the model
4. Access visualizations via `results['qN']['model'].visualize_topics()`, etc.
5. Check topic info with `results['qN']['model'].get_topic_info()`

Example tuning workflow for Q1:
```python
# Override UMAP for Q1
umap_q1 = UMAP(n_neighbors=10, n_components=5, min_dist=0.1, metric="cosine", random_state=42)
model_q1 = BERTopic(..., umap_model=umap_q1, ...)
topics_q1, probs_q1 = model_q1.fit_transform(docs_q1)
results['q1'] = {'model': model_q1, 'topics': topics_q1, 'probs': probs_q1, 'docs': docs_q1}
```

In [20]:
# Parameter summary table for Q1-Q11
import pandas as pd

parameter_rows = [
    {"question": "Q1",  "column_index": 13, "vectorizer_min_df": 5, "vectorizer_max_df": 0.7, "ngram_range": "(1,2)", "mmr_diversity": 0.9, "umap_n_neighbors": 10, "umap_n_components": 10, "hdbscan_min_cluster_size": 15, "hdbscan_min_samples": 3},
    {"question": "Q2",  "column_index": 8,  "vectorizer_min_df": 3, "vectorizer_max_df": 0.7, "ngram_range": "(1,2)", "mmr_diversity": 0.3, "umap_n_neighbors": 10, "umap_n_components": 10, "hdbscan_min_cluster_size": 20, "hdbscan_min_samples": 3},
    {"question": "Q3",  "column_index": 10, "vectorizer_min_df": 3, "vectorizer_max_df": 0.7, "ngram_range": "(1,2)", "mmr_diversity": 0.3, "umap_n_neighbors": 18, "umap_n_components": 5,  "hdbscan_min_cluster_size": 15, "hdbscan_min_samples": 3},
    {"question": "Q4",  "column_index": 11, "vectorizer_min_df": 3, "vectorizer_max_df": 0.7, "ngram_range": "(1,2)", "mmr_diversity": 0.3, "umap_n_neighbors": 18, "umap_n_components": 5,  "hdbscan_min_cluster_size": 15, "hdbscan_min_samples": 3},
    {"question": "Q5",  "column_index": 12, "vectorizer_min_df": 3, "vectorizer_max_df": 0.7, "ngram_range": "(1,2)", "mmr_diversity": 0.3, "umap_n_neighbors": 18, "umap_n_components": 5,  "hdbscan_min_cluster_size": 15, "hdbscan_min_samples": 3},
    {"question": "Q6",  "column_index": 9,  "vectorizer_min_df": 1, "vectorizer_max_df": 1.0, "ngram_range": "(1,2)", "mmr_diversity": 0.3, "umap_n_neighbors": 18, "umap_n_components": 5,  "hdbscan_min_cluster_size": 12, "hdbscan_min_samples": 2},
    {"question": "Q7",  "column_index": 4,  "vectorizer_min_df": 1, "vectorizer_max_df": 1.0, "ngram_range": "(1,2)", "mmr_diversity": 0.3, "umap_n_neighbors": 22, "umap_n_components": 5,  "hdbscan_min_cluster_size": 18, "hdbscan_min_samples": 3},
    {"question": "Q8",  "column_index": 14, "vectorizer_min_df": 1, "vectorizer_max_df": 1.0, "ngram_range": "(1,2)", "mmr_diversity": 0.3, "umap_n_neighbors": 15, "umap_n_components": 5,  "hdbscan_min_cluster_size": 12, "hdbscan_min_samples": 2},
    {"question": "Q9",  "column_index": 16, "vectorizer_min_df": 1, "vectorizer_max_df": 1.0, "ngram_range": "(1,2)", "mmr_diversity": 0.3, "umap_n_neighbors": 12, "umap_n_components": 5,  "hdbscan_min_cluster_size": 10, "hdbscan_min_samples": 2},
    {"question": "Q10", "column_index": 17, "vectorizer_min_df": 3, "vectorizer_max_df": 0.7, "ngram_range": "(1,2)", "mmr_diversity": 0.3, "umap_n_neighbors": 18, "umap_n_components": 5,  "hdbscan_min_cluster_size": 15, "hdbscan_min_samples": 3},
    {"question": "Q11", "column_index": 15, "vectorizer_min_df": 3, "vectorizer_max_df": 0.7, "ngram_range": "(1,2)", "mmr_diversity": 0.3, "umap_n_neighbors": 18, "umap_n_components": 5,  "hdbscan_min_cluster_size": 15, "hdbscan_min_samples": 3},
]

parameter_table = pd.DataFrame(parameter_rows)
display(parameter_table)

,question,column_index,vectorizer_min_df,vectorizer_max_df,ngram_range,mmr_diversity,umap_n_neighbors,umap_n_components,hdbscan_min_cluster_size,hdbscan_min_samples
0,Q1,13,5,0.7,"(1,2)",0.9,10,10,15,3
1,Q2,8,3,0.7,"(1,2)",0.3,10,10,20,3
2,Q3,10,3,0.7,"(1,2)",0.3,18,5,15,3
3,Q4,11,3,0.7,"(1,2)",0.3,18,5,15,3
4,Q5,12,3,0.7,"(1,2)",0.3,18,5,15,3
5,Q6,9,1,1.0,"(1,2)",0.3,18,5,12,2
6,Q7,4,1,1.0,"(1,2)",0.3,22,5,18,3
7,Q8,14,1,1.0,"(1,2)",0.3,15,5,12,2
8,Q9,16,1,1.0,"(1,2)",0.3,12,5,10,2
9,Q10,17,3,0.7,"(1,2)",0.3,18,5,15,3


# Visualizations

In [21]:
model_q1.visualize_documents(docs_q1, hide_annotations=True)

In [22]:
model_q2.visualize_documents(docs_q2, hide_annotations=True)

In [23]:
model_q3.visualize_documents(docs_q3, hide_annotations=True)

In [24]:
model_q4.visualize_documents(docs_q4, hide_annotations=True)

In [25]:
model_q5.visualize_documents(docs_q5, hide_annotations=True)

In [26]:
model_q6.visualize_documents(docs_q6, hide_annotations=True)

In [27]:
model_q7.visualize_documents(docs_q7, hide_annotations=True)

In [28]:
model_q8.visualize_documents(docs_q8, hide_annotations=True)

In [29]:
model_q9.visualize_documents(docs_q9, hide_annotations=True)

In [30]:
model_q10.visualize_documents(docs_q10, hide_annotations=True)

In [31]:
model_q11.visualize_documents(docs_q11, hide_annotations=True)

# Saving results

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Export BERTopic outputs for every fitted question model in `results`.
output_root = Path("../data/bertopic_exports/per_question")
output_root.mkdir(parents=True, exist_ok=True)

# Set to True if you also want serialized BERTopic model folders (large on disk).
save_serialized_models = False

summary_rows = []

for question, payload in results.items():
    model = payload["model"]
    docs = payload["docs"]
    topics = payload.get("topics")
    probs = payload.get("probs")

    q_dir = output_root / question
    q_dir.mkdir(parents=True, exist_ok=True)

    # 1) Topic-level summary
    topic_info = model.get_topic_info()
    topic_info.to_csv(q_dir / "topic_info.csv", index=False)

    # 2) Document-level assignments/probabilities
    doc_info = model.get_document_info(docs)
    doc_info.to_csv(q_dir / "document_info.csv", index=False)

    # 3) Full topic-word representation (long format)
    topic_rows = []
    for topic_id, words in model.get_topics().items():
        if words is None:
            continue
        for rank, (word, score) in enumerate(words, start=1):
            topic_rows.append(
                {
                    "topic": topic_id,
                    "rank": rank,
                    "word": word,
                    "score": float(score),
                }
            )
    pd.DataFrame(topic_rows).to_csv(q_dir / "topic_words_long.csv", index=False)

    # 4) Full document-topic probabilities (if available)
    probs_exported = False
    if probs is not None:
        probs_array = np.asarray(probs)
        if probs_array.ndim == 1:
            probs_array = probs_array.reshape(-1, 1)

        # BERTopic probabilities usually map to non-outlier topics only.
        topic_ids = sorted(t for t in model.get_topics().keys() if t != -1)
        if probs_array.shape[1] == len(topic_ids):
            prob_columns = [f"topic_{topic_id}" for topic_id in topic_ids]
        else:
            prob_columns = [f"topic_index_{i}" for i in range(probs_array.shape[1])]

        probs_wide = pd.DataFrame(probs_array, columns=prob_columns)
        probs_wide.insert(0, "document_index", np.arange(len(probs_wide)))
        probs_wide.insert(1, "document", docs[:len(probs_wide)])
        probs_wide.to_csv(q_dir / "document_topic_probabilities_wide.csv", index=False)

        probs_long = probs_wide.melt(
            id_vars=["document_index", "document"],
            var_name="topic",
            value_name="probability",
        )
        probs_long.to_csv(q_dir / "document_topic_probabilities_long.csv", index=False)
        probs_exported = True

    # 5) Useful metadata for quick auditing
    n_docs = len(docs)
    n_outliers = int(sum(t == -1 for t in topics)) if topics is not None else None
    n_topics_excl_outlier = int(topic_info[topic_info["Topic"] != -1].shape[0])
    metadata = {
        "question": question,
        "n_documents": n_docs,
        "n_outliers": n_outliers,
        "outlier_rate": (n_outliers / n_docs) if (n_outliers is not None and n_docs > 0) else None,
        "n_topics_excluding_outlier": n_topics_excl_outlier,
        "has_probabilities": probs is not None,
        "exported_document_topic_probabilities": probs_exported,
    }
    (q_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))

    if save_serialized_models:
        model.save(str(q_dir / "bertopic_model"), save_embedding_model=False)

    summary_rows.append(metadata)

summary_df = pd.DataFrame(summary_rows).sort_values("question")
summary_df.to_csv(output_root / "bertopic_export_summary.csv", index=False)

print(f"Saved BERTopic exports to: {output_root.resolve()}")
display(summary_df)

Saved BERTopic exports to: /Users/tylerwiederich/Library/CloudStorage/OneDrive-UniversityofNebraska-Lincoln/4 - Obsidian Vault/Research/dissertation/ch2-experiential-learning/data/bertopic_exports/per_question


,question,n_documents,n_outliers,outlier_rate,n_topics_excluding_outlier,has_probabilities,exported_document_topic_probabilities
0,q1,632,172,0.272152,11,True,True
9,q10,405,48,0.118519,5,True,True
10,q11,405,4,0.009877,5,True,True
1,q2,521,47,0.090211,7,True,True
2,q3,517,135,0.261122,6,True,True
3,q4,517,129,0.249516,8,True,True
4,q5,515,90,0.174757,10,True,True
5,q6,514,61,0.118677,10,True,True
6,q7,495,107,0.216162,10,True,True
7,q8,405,61,0.150617,11,True,True


: 